# Catering Problem

O problema do serviço de catering surge quando é necessário programar (agendar) os serviços de uma empresa fornecedora de refeições. Ele se baseia em um artigo clássico de **William Prager (1903–1980)**. A versão discutida é formulada como um **modelo de transbordo (transshipment)** e resolvida por meio do **algoritmo simplex de rede**.

## Apresentação do problema

Uma empresa de catering fornece refeições conforme pedidos dos clientes. A empresa contratou seus serviços para **sete dias consecutivos**; esses sete dias formam o horizonte de planejamento. A cada refeição, a empresa fornece um **guardanapo limpo**. A demanda de refeições (e, portanto, de guardanapos) ao longo do período de planejamento é conhecida e está listada na **Tabela 1**.

Após o uso pelo cliente, o guardanapo é colocado no **cesto de guardanapos sujos**, que são enviados à lavanderia. A lavanderia oferece dois tipos de serviço:

- **Serviço rápido (fast):** lavagem em **2 dias**, ao custo de **US$ 0,75** por guardanapo.
- **Serviço lento (slow):** lavagem em **4 dias**, ao custo de **US$ 0,50** por guardanapo.

Os guardanapos sujos não precisam ser enviados imediatamente para a lavanderia: o caterer pode optar por **carregá-los para o dia seguinte** (ou mais à frente). Assim, o cesto de guardanapos sujos no dia $j$ (com $j = 1, \dots, 7$) pode conter guardanapos usados no próprio dia $j$ e também guardanapos usados em dias anteriores.

Quando o número de guardanapos que retornam da lavanderia em determinado dia não é suficiente para atender à demanda, o caterer pode **comprar guardanapos novos** ao preço de **US$ 3,00** por unidade.

### Tabela 1 — Número de guardanapos demandados

| Dia | Guardanapos demandados |
|:---:|:---:|
| 1 | 23 |
| 2 | 14 |
| 3 | 19 |
| 4 | 21 |
| 5 | 18 |
| 6 | 14 |
| 7 | 15 |

No início de cada dia, a empresa precisa decidir quantos guardanapos novos comprar para atender à demanda, considerando os guardanapos que retornam da lavanderia. Ao final de cada dia, são tomadas quatro decisões sobre o estoque de guardanapos sujos:

- Quantos guardanapos sujos enviar ao **serviço lento** da lavanderia?
- Quantos guardanapos sujos enviar ao **serviço rápido** da lavanderia?
- Quantos guardanapos sujos **carregar para o dia seguinte**?
- Quantos guardanapos **novos comprar**?

O objetivo é encontrar uma sequência de decisões que **minimize o custo total**. Assume-se que o caterer **não possui guardanapos** (limpos ou sujos) no início do período de planejamento e que todos os guardanapos demandados podem ser comprados.

## Formulação Tradicional

### Bibliotecas

In [1]:
from ortools.linear_solver import pywraplp

### Instanciação do solver

In [2]:
solver = pywraplp.Solver.CreateSolver("SCIP")

### Dados

In [3]:
DEMANDAS = {
    1 : 23,
    2 : 14,
    3 : 19,
    4 : 21,
    5 : 18,
    6 : 14,
    7 : 15
}

DIAS_LAVAGEM_RAPIDA = 2

VALOR_LAVAGEM_RAPIDA = 0.75

DIAS_LAVAGEM_LENTA = 4

VALOR_LAVAGEM_LENTA = 0.5

VALOR_COMPRA = 3

# fast, slow, retain, new
SITUACOES = 'f','s','r','n'

### Variáveis

x1f (guardanapos sujos encaminhados para a lavagem rápida da lavanderia no final do 1º dia) (fast)

x1s (guardanapos sujos encaminhados para a lavagem lenta da lavanderia no final do 1º dia)  (slow)

x1r (guardanapos sujos armazenados ao final do 1º dia) (to retain)

x1n (guardanapos novos comprados no início do 1º dia) (new)

**existem 28 variáveis (7 para cada dia x 4 para cada situação)**

In [4]:
variaveis = list()

for i in DEMANDAS.keys():
    for s in SITUACOES:
        variaveis.append(solver.IntVar(0, solver.infinity(), f'x_{i}_{s}'))

### Função objetivo

**Minimizar** -> 0,75 * x1f + 0,5 * x1s + 3 * x1n + ... + 0,75 * x7f + 0,5 * x7s + 3 * x7n

In [5]:
objetivo = solver.Objective()

for var in variaveis:
    match var.name()[-1]:
        case 'f':
            objetivo.SetCoefficient(var, VALOR_LAVAGEM_RAPIDA)
        case 's':
            objetivo.SetCoefficient(var, VALOR_LAVAGEM_LENTA)
        case 'n':
            objetivo.SetCoefficient(var, VALOR_COMPRA)

objetivo.SetMinimization()

### Restrições

In [6]:
dicionario_variaveis = {var.name(): var for var in variaveis}

def get_var(i, s):
    return dicionario_variaveis[f'x_{i}_{s}']

# 1) Demanda de cada dia: comprados + chegados da lavanderia = demanda (para garantir que a demanda seja exatamente atendida)
for i, d in DEMANDAS.items():
    ct = solver.Constraint(d, d, f'demanda_dia_{i}')
    ct.SetCoefficient(get_var(i, 'n'), 1)
    if i - DIAS_LAVAGEM_RAPIDA >= 1:
        ct.SetCoefficient(get_var(i - DIAS_LAVAGEM_RAPIDA, 'f'), 1)
    if i - DIAS_LAVAGEM_LENTA >= 1:
        ct.SetCoefficient(get_var(i - DIAS_LAVAGEM_LENTA, 's'), 1)

# 2) Balanço do cesto de sujos: (para garantir que nenhum guardanapo surja do nada ou se perca)
#    sujos_anterior + demanda_hoje = fast_hoje + slow_hoje + retidos_hoje
for i in DEMANDAS.keys():
    ct = solver.Constraint(DEMANDAS[i], DEMANDAS[i], f'balanco_sujos_dia_{i}')
    ct.SetCoefficient(get_var(i, 'f'),  1)
    ct.SetCoefficient(get_var(i, 's'),  1)
    ct.SetCoefficient(get_var(i, 'r'),  1)
    if i - 1 >= 1:
        ct.SetCoefficient(get_var(i - 1, 'r'), -1)

In [13]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 14
\   Variables        : 28
\     Binary         : 0
\     Integer        : 28
\     Continuous     : 0
Minimize
 Obj: +0.75 x_1_f +0.5 x_1_s +3 x_1_n +0.75 x_2_f +0.5 x_2_s +3 x_2_n +0.75 x_3_f +0.5 x_3_s +3 x_3_n +0.75 x_4_f +0.5 x_4_s +3 x_4_n +0.75 x_5_f +0.5 x_5_s +3 x_5_n +0.75 x_6_f +0.5 x_6_s +3 x_6_n +0.75 x_7_f +0.5 x_7_s +3 x_7_n 
Subject to
 demanda_dia_1: +1 x_1_n  = 23
 demanda_dia_2: +1 x_2_n  = 14
 demanda_dia_3: +1 x_1_f +1 x_3_n  = 19
 demanda_dia_4: +1 x_2_f +1 x_4_n  = 21
 demanda_dia_5: +1 x_1_s +1 x_3_f +1 x_5_n  = 18
 demanda_dia_6: +1 x_2_s +1 x_4_f +1 x_6_n  = 14
 demanda_dia_7: +1 x_3_s +1 x_5_f +1 x_7_n  = 15
 balanco_sujos_dia_1: +1 x_1_f +1 x_1_s +1 x_1_r  = 23
 balanco_sujos_dia_2: -1 x_1_r +1 x_2_f +1 x_2_s +1 x_2_r  = 14
 balanco_sujos_dia_3: -1 x_2_r +1 x_3_f +1 x_3_s +1 x_3_r  = 19
 balanco_sujos_dia_4: -1 x_3_r +1 x_4_f +1 x_4_s +1 x_4_r  = 

### Resolução e Resultados

In [8]:
status = solver.Solve()

In [9]:
if status == pywraplp.Solver.OPTIMAL:
    print('Solução ótima encontrada!')
elif status == pywraplp.Solver.FEASIBLE:
    print('Solução viável encontrada, mas não necessariamente ótima.')
elif status == pywraplp.Solver.INFEASIBLE:
    print('Problema inviável.')
elif status == pywraplp.Solver.UNBOUNDED:
    print('Problema ilimitado.')
else:
    print('Solver não conseguiu resolver.')

Solução ótima encontrada!


In [10]:
f'Custo total: US$ {solver.Objective().Value():.2f}'

'Custo total: US$ 182.75'

In [11]:
for var in variaveis:
    print(f'{var.name()} = {var.solution_value()}')

x_1_f = 18.0
x_1_s = 0.0
x_1_r = 5.0
x_1_n = 23.0
x_2_f = 19.0
x_2_s = 0.0
x_2_r = 0.0
x_2_n = 14.0
x_3_f = 18.0
x_3_s = 1.0
x_3_r = 0.0
x_3_n = 1.0
x_4_f = 14.0
x_4_s = 0.0
x_4_r = 7.0
x_4_n = 2.0
x_5_f = 14.0
x_5_s = 0.0
x_5_r = 11.0
x_5_n = 0.0
x_6_f = 0.0
x_6_s = -0.0
x_6_r = 25.0
x_6_n = 0.0
x_7_f = 0.0
x_7_s = -0.0
x_7_r = 40.0
x_7_n = 0.0


In [15]:
if status == pywraplp.Solver.OPTIMAL:
    print(f'Custo total: US$ {solver.Objective().Value():.2f}\n')
    
    nomes_situacao = {
        'n': 'Comprados',
        'f': 'Enviados p/ rápida',
        's': 'Enviados p/ lenta',
        'r': 'Retidos p/ amanhã',
    }
    
    print(f'{"Dia":<5}' + ''.join(f'{nome:<25}' for nome in nomes_situacao.values()))

    for i in DEMANDAS.keys():
        valores = [
            f'{int(dicionario_variaveis[f"x_{i}_{s}"].solution_value()):<25}'
            for s in nomes_situacao.keys()
        ]
        print(f'{i:<5}' + ''.join(valores))

Custo total: US$ 182.75

Dia  Comprados                Enviados p/ rápida       Enviados p/ lenta        Retidos p/ amanhã        
1    23                       18                       0                        5                        
2    14                       19                       0                        0                        
3    1                        18                       1                        0                        
4    2                        14                       0                        7                        
5    0                        14                       0                        11                       
6    0                        0                        0                        25                       
7    0                        0                        0                        40                       


## Formulação como Problema de Transbordo

O problema do catering pode ser representado como um **problema de fluxo em rede** do tipo transbordo, onde guardanapos "fluem" através de um grafo direcionado, partindo de fontes (oferta) até sumidouros (demanda), com custos associados a cada arco.

### Nós da rede

Definem-se dois conjuntos de nós:

**Fontes (oferta de guardanapos):** $\{O_0, O_1, O_2, O_3, O_4, O_5, O_6, O_7\}$

- $O_0$ é o **fornecedor de guardanapos novos**, com estoque ilimitado (na prática, basta um valor grande o suficiente, como 125).
- $O_j$ (para $j = 1, \dots, 7$) representa o **cesto de guardanapos sujos ao final do dia $j$**. Cada $O_j$ "oferta" exatamente $d_j$ guardanapos sujos (a demanda do dia, pois cada refeição servida gera um sujo).

**Sumidouros (demanda de guardanapos):** $\{D_0, D_1, D_2, D_3, D_4, D_5, D_6, D_7\}$

- $D_j$ (para $j = 1, \dots, 7$) é o **ponto de consumo do dia $j$**, com demanda $d_j$.
- $D_0$ é um **sumidouro artificial** que absorve, a custo zero, todos os guardanapos que sobrarem na rede ao final do planejamento (sujos não lavados, ou compras excedentes). Sua presença garante que $\sum b_i = 0$, condição necessária para um problema de transbordo balanceado.

### Arcos da rede

Há quatro tipos de arcos, cada um representando uma decisão:

| Tipo de arco | Conecta | Custo | Variável |
|---|---|---|---|
| Compra de novos | $O_0 \to D_j$ | $\$3{,}00$ | $p_j$ |
| Lavagem rápida (2 dias) | $O_j \to D_{j+2}$ | $\$0{,}75$ | $f_j$ |
| Lavagem lenta (4 dias) | $O_j \to D_{j+4}$ | $\$0{,}50$ | $s_j$ |
| Carregar adiante | $O_j \to O_{j+1}$ | $\$0{,}00$ | $h_j$ |
| Descarte final | $O_0 \to D_0$, $O_7 \to D_0$ | $\$0{,}00$ | $p_0$, $h_7$ |

> **Observação 1:** Os arcos de lavagem só existem se a lavagem retornar dentro do horizonte. Por isso, $f_j$ existe apenas para $j = 1, \dots, 5$ (lavagem rápida iniciada após o dia 5 não retorna), e $s_j$ apenas para $j = 1, 2, 3$ (lavagem lenta após o dia 3 não retorna).

> **Observação 2:** 125 é o valor escolhido pelo livro para representar "estoque suficiente para o caterer comprar quantos guardanapos precisar". Como a demanda total ao longo dos 7 dias é $23+14+19+21+18+14+15=124$, qualquer valor maior ou igual a isso funciona. O livro escolheu 125, um a mais que o máximo possível necessário, garantindo, dessa forma, viabilidade.

### Diagrama da rede

```mermaid
graph LR
    O0(("O₀<br/>+125"))
    O1(("O₁<br/>+23"))
    O2(("O₂<br/>+14"))
    O3(("O₃<br/>+19"))
    O4(("O₄<br/>+21"))
    O5(("O₅<br/>+18"))
    O6(("O₆<br/>+14"))
    O7(("O₇<br/>+15"))

    D0(("D₀<br/>-125"))
    D1(("D₁<br/>-23"))
    D2(("D₂<br/>-14"))
    D3(("D₃<br/>-19"))
    D4(("D₄<br/>-21"))
    D5(("D₅<br/>-18"))
    D6(("D₆<br/>-14"))
    D7(("D₇<br/>-15"))

    %% Compras (O0 -> Dj), custo $3
    O0 -- "p₁ [3]" --> D1
    O0 -- "p₂ [3]" --> D2
    O0 -- "p₃ [3]" --> D3
    O0 -- "p₄ [3]" --> D4
    O0 -- "p₅ [3]" --> D5
    O0 -- "p₆ [3]" --> D6
    O0 -- "p₇ [3]" --> D7
    O0 -- "p₀ [0]" --> D0

    %% Lavagem rápida (Oj -> D_{j+2}), custo $0.75
    O1 -- "f₁ [0,75]" --> D3
    O2 -- "f₂ [0,75]" --> D4
    O3 -- "f₃ [0,75]" --> D5
    O4 -- "f₄ [0,75]" --> D6
    O5 -- "f₅ [0,75]" --> D7

    %% Lavagem lenta (Oj -> D_{j+4}), custo $0.50
    O1 -- "s₁ [0,50]" --> D5
    O2 -- "s₂ [0,50]" --> D6
    O3 -- "s₃ [0,50]" --> D7

    %% Carregar adiante (Oj -> O_{j+1}), custo 0
    O1 -- "h₁ [0]" --> O2
    O2 -- "h₂ [0]" --> O3
    O3 -- "h₃ [0]" --> O4
    O4 -- "h₄ [0]" --> O5
    O5 -- "h₅ [0]" --> O6
    O6 -- "h₆ [0]" --> O7
    O7 -- "h₇ [0]" --> D0

    classDef fonte fill:#cde4ff,stroke:#1e40af,stroke-width:2px,color:#000;
    classDef sumidouro fill:#ffe4cd,stroke:#b45309,stroke-width:2px,color:#000;
    class O0,O1,O2,O3,O4,O5,O6,O7 fonte;
    class D0,D1,D2,D3,D4,D5,D6,D7 sumidouro;
```

**Legenda:**

- Nós azuis: fontes $O_j$ (oferta de guardanapos).
- Nós laranja: sumidouros $D_j$ (demanda de guardanapos).
- Rótulos dos arcos: variável de fluxo e, entre colchetes, o custo por unidade.

### Vetor de oferta-demanda

Cada nó tem associado um valor $b_i$ que representa sua oferta (positivo) ou demanda (negativo):

$$
\mathbf{b} = [\underbrace{125}_{O_0},\ \underbrace{23, 14, 19, 21, 18, 14, 15}_{O_1, \dots, O_7},\ \underbrace{-125}_{D_0},\ \underbrace{-23, -14, -19, -21, -18, -14, -15}_{D_1, \dots, D_7}]^\top
$$

A soma é zero, conforme exigido em um problema de transbordo balanceado.

### Relação com a formulação do notebook

A modelagem implementada em Python usa nomes ligeiramente diferentes, mas as variáveis correspondem **diretamente** às do modelo de transbordo do livro:

| Notebook | Transbordo do livro | Tipo de arco | Significado |
|---|---|---|---|
| $x_{j,n}$ | $p_j$ | $O_0 \to D_j$ | comprados no dia $j$ |
| $x_{j,f}$ | $f_j$ | $O_j \to D_{j+2}$ | enviados à lavagem rápida no dia $j$ |
| $x_{j,s}$ | $s_j$ | $O_j \to D_{j+4}$ | enviados à lavagem lenta no dia $j$ |
| $x_{j,r}$ | $h_j$ | $O_j \to O_{j+1}$ | retidos (carregados) ao final do dia $j$ |

São $7 \times 4 = 28$ variáveis no total — uma para cada par (dia, tipo de decisão). Note que os arcos auxiliares $p_0$ (excesso de novos não comprados) e $h_7$ (sujos descartados no fim do horizonte) são representados implicitamente: como têm custo zero e existem apenas para balancear a rede, o modelo do notebook prescinde deles ao usar diretamente o balanço local do dia.

### Restrições como conservação de fluxo

Em um problema de transbordo, cada nó tem uma **restrição de conservação de fluxo**: o que entra é igual ao que sai (descontando oferta/demanda do próprio nó). No notebook, essa correspondência se traduz em dois conjuntos de restrições:

- **Em cada sumidouro $D_j$ (dia $j = 1, \dots, 7$) — restrição `demanda_dia_j`:**
  Tudo que entra (compras + lavagens que retornam) deve igualar a demanda $d_j$.
  $$
  p_j + f_{j-2} + s_{j-4} = d_j
  $$
  (termos com índice $< 1$ são omitidos)

- **Em cada fonte $O_j$ (dia $j = 1, \dots, 7$) — restrição `balanco_sujos_dia_j`:**
  Tudo que sai (lavagens iniciadas + retidos para amanhã) deve igualar a oferta de sujos (demanda do dia + retidos de ontem).
  $$
  f_j + s_j + h_j = d_j + h_{j-1}
  $$
  (com $h_0 = 0$)

As restrições de $O_0$ e $D_0$ do modelo do livro ($p_0 + p_1 + \dots + p_7 = 125$ e $p_0 + h_7 = 125$) **não precisam aparecer explicitamente** no notebook: como $p_0$ é livre (sem custo) e o estoque de $O_0$ é superdimensionado, ele se ajusta automaticamente ao residual. O mesmo vale para $h_7$, que apenas absorve sujos do fim do horizonte.

### Função objetivo

Minimizar o custo total dos arcos efetivamente usados:

$$
\min \quad 3 \sum_{j=1}^{7} p_j \;+\; 0{,}75 \sum_{j=1}^{5} f_j \;+\; 0{,}50 \sum_{j=1}^{3} s_j
$$

Arcos de custo zero (carregar adiante $h_j$, descarte $p_0$ e $h_7$) não contribuem para a função objetivo.

## Aplicação do Algoritmo Simplex em Rede

Nesta seção, aplicamos o **algoritmo simplex em rede** à modelagem de transbordo do problema do catering. A ideia é caminhar de **solução-árvore viável** em **solução-árvore viável**, sempre reduzindo o custo total, até que não exista mais nenhum arco fora da árvore capaz de melhorar o custo.

> **Nota sobre a notação:** usamos $O_j$ para os nós de oferta (originalmente $P_j$ no livro) e $D_j$ para os nós de demanda (originalmente $Q_j$). As variáveis de fluxo seguem as mesmas: $p_j$ (compras), $f_j$ (lavagem rápida), $s_j$ (lavagem lenta) e $h_j$ (carregar adiante).

---

### Inicialização

Uma **solução-árvore viável inicial** é obtida de forma trivial: comprar todos os guardanapos demandados a cada dia e manter os sujos até o final do horizonte de planejamento, sem enviar nada à lavanderia.

Os valores das variáveis básicas correspondentes a essa solução são:

| Variável | Valor | Variável | Valor |
|:---:|:---:|:---:|:---:|
| $p_0$ | 1   | $p_1$ | 23 |
| $p_2$ | 14  | $p_3$ | 19 |
| $p_4$ | 21  | $p_5$ | 18 |
| $p_6$ | 14  | $p_7$ | 15 |
| $h_1$ | 23  | $h_2$ | 37 |
| $h_3$ | 56  | $h_4$ | 77 |
| $h_5$ | 95  | $h_6$ | 109 |
| $h_7$ | 124 | | |

Custo dessa solução inicial: $124 \times \$3 = \$372{,}00$.

Note que essa solução-árvore tem 15 variáveis básicas, o que corresponde ao número de nós menos um ($16 - 1 = 15$) — uma propriedade característica de soluções-árvore em redes de transbordo.

**Árvore inicial:**

```mermaid
graph LR
    O0(("O₀"))
    O1(("O₁"))
    O2(("O₂"))
    O3(("O₃"))
    O4(("O₄"))
    O5(("O₅"))
    O6(("O₆"))
    O7(("O₇"))

    D0(("D₀"))
    D1(("D₁"))
    D2(("D₂"))
    D3(("D₃"))
    D4(("D₄"))
    D5(("D₅"))
    D6(("D₆"))
    D7(("D₇"))

    %% Compras de O0 para cada Dj
    O0 -- "p₁=23" --> D1
    O0 -- "p₂=14" --> D2
    O0 -- "p₃=19" --> D3
    O0 -- "p₄=21" --> D4
    O0 -- "p₅=18" --> D5
    O0 -- "p₆=14" --> D6
    O0 -- "p₇=15" --> D7
    O0 -- "p₀=1" --> D0

    %% Cadeia de retenção (todos os sujos seguram até o fim)
    O1 -- "h₁=23" --> O2
    O2 -- "h₂=37" --> O3
    O3 -- "h₃=56" --> O4
    O4 -- "h₄=77" --> O5
    O5 -- "h₅=95" --> O6
    O6 -- "h₆=109" --> O7
    O7 -- "h₇=124" --> D0

    classDef fonte fill:#cde4ff,stroke:#1e40af,stroke-width:2px,color:#000;
    classDef sumidouro fill:#ffe4cd,stroke:#b45309,stroke-width:2px,color:#000;
    class O0,O1,O2,O3,O4,O5,O6,O7 fonte;
    class D0,D1,D2,D3,D4,D5,D6,D7 sumidouro;
```

---

### Iteração 1

Seja $[y_{O_0},\ \dots,\ y_{O_7},\ y_{D_0},\ \dots,\ y_{D_7}]$ o vetor de **variáveis duais** (uma por nó). Escolhemos $O_0$ como raiz da árvore e fixamos $y_{O_0} = 0$. Os demais valores são obtidos propagando pela árvore, segundo a regra do simplex em rede: para cada arco básico $(i, j)$ com custo $c_{ij}$, vale $y_i - y_j = c_{ij}$.

Os valores duais resultantes são:

$$
\begin{aligned}
y_{O_j} &= 0 \quad \text{para todo } j = 0, 1, \dots, 7 \\
y_{D_0} &= 0 \\
y_{D_j} &= -3 \quad \text{para } j = 1, \dots, 7
\end{aligned}
$$

As variáveis **não-básicas** são $f_1, f_2, f_3, f_4, f_5, s_1, s_2, s_3$ (as únicas que ainda não foram usadas). Seus **custos duais reduzidos** são:

$$
\bar c_{ij} = c_{ij} - (y_i - y_j)
$$

Calculando:

$$
\begin{bmatrix} \bar c_{f_1} & \bar c_{f_2} & \bar c_{f_3} & \bar c_{f_4} & \bar c_{f_5} & \bar c_{s_1} & \bar c_{s_2} & \bar c_{s_3} \end{bmatrix}
= \begin{bmatrix} -2{,}25 & -2{,}25 & -2{,}25 & -2{,}25 & -2{,}25 & -2{,}50 & -2{,}50 & -2{,}50 \end{bmatrix}
$$

Todos os custos reduzidos são **negativos**, indicando que qualquer um deles, se introduzido na base, reduzirá o custo total. Escolhemos arbitrariamente o de menor valor: o arco $(O_1, D_5)$, correspondente a $s_1$, como **arco entrante**.

A inserção desse arco forma o ciclo:

$$
D_5 \to O_0 \to D_0 \to O_7 \to O_6 \to O_5 \to O_4 \to O_3 \to O_2 \to O_1
$$

O único arco do ciclo que aponta no mesmo sentido do arco entrante é $(O_0, D_0)$, correspondente a $p_0$. Aumentamos o fluxo de $s_1$ até que o fluxo em $(O_0, D_5)$ (correspondente a $p_5$) chegue a zero — $p_5$ é o primeiro a zerar, e portanto **sai da base**.

Os novos valores básicos são:

| Variável | Valor | Variável | Valor |
|:---:|:---:|:---:|:---:|
| $p_0$ | 19 | $p_1$ | 23 |
| $p_2$ | 14 | $p_3$ | 19 |
| $p_4$ | 21 | $p_6$ | 14 |
| $p_7$ | 15 | $s_1$ | 18 |
| $h_1$ | 5  | $h_2$ | 19 |
| $h_3$ | 38 | $h_4$ | 59 |
| $h_5$ | 77 | $h_6$ | 91 |
| $h_7$ | 106 | | |

Novo custo:

- Compras: $106 \times \$3 = \$318{,}00$
- Lavagem lenta: $18 \times \$0{,}50 = \$9{,}00$
- **Total: \$327,00**

**Árvore após a Iteração 1:**

```mermaid
graph LR
    O0(("O₀"))
    O1(("O₁"))
    O2(("O₂"))
    O3(("O₃"))
    O4(("O₄"))
    O5(("O₅"))
    O6(("O₆"))
    O7(("O₇"))

    D0(("D₀"))
    D1(("D₁"))
    D2(("D₂"))
    D3(("D₃"))
    D4(("D₄"))
    D5(("D₅"))
    D6(("D₆"))
    D7(("D₇"))

    %% Compras (p5 saiu da base)
    O0 -- "p₁=23" --> D1
    O0 -- "p₂=14" --> D2
    O0 -- "p₃=19" --> D3
    O0 -- "p₄=21" --> D4
    O0 -- "p₆=14" --> D6
    O0 -- "p₇=15" --> D7
    O0 -- "p₀=19" --> D0

    %% Lavagem lenta (s1 entrou na base)
    O1 -- "s₁=18" --> D5

    %% Cadeia de retenção
    O1 -- "h₁=5"  --> O2
    O2 -- "h₂=19" --> O3
    O3 -- "h₃=38" --> O4
    O4 -- "h₄=59" --> O5
    O5 -- "h₅=77" --> O6
    O6 -- "h₆=91" --> O7
    O7 -- "h₇=106" --> D0

    classDef fonte fill:#cde4ff,stroke:#1e40af,stroke-width:2px,color:#000;
    classDef sumidouro fill:#ffe4cd,stroke:#b45309,stroke-width:2px,color:#000;
    classDef entrou fill:#d1fae5,stroke:#059669,stroke-width:3px,color:#000;
    class O0,O1,O2,O3,O4,O5,O6,O7 fonte;
    class D0,D1,D2,D3,D4,D5,D6,D7 sumidouro;
    class D5 entrou;
```

> O nó $D_5$ aparece destacado em verde porque foi alcançado pelo arco entrante $s_1$ nesta iteração.

---

### Iteração 2

Atualizamos as variáveis duais. Como o arco entrante foi $(O_1, D_5)$ com $c = 0{,}50$, e a árvore agora liga $D_5$ a $O_1$ via esse arco, temos $y_{O_1} - y_{D_5} = 0{,}50$. Como $y_{O_1} = 0$:

$$
y_{D_5} = -\tfrac{1}{2}
$$

Os demais valores permanecem inalterados. Os novos custos duais reduzidos das variáveis fora da árvore são:

$$
\begin{bmatrix} \bar c_{f_1} & \bar c_{f_2} & \bar c_{f_3} & \bar c_{f_4} & \bar c_{f_5} & \bar c_{s_1} & \bar c_{s_2} & \bar c_{s_3} \end{bmatrix}
= \begin{bmatrix} 2{,}50 & -2{,}25 & -2{,}25 & 0{,}25 & -2{,}25 & -2{,}25 & -2{,}50 & -2{,}50 \end{bmatrix}
$$

Note que $\bar c_{f_1}$ deixou de ser negativo (não vale a pena enviar à lavagem rápida no dia 1 nesta configuração), enquanto $\bar c_{s_2}$ e $\bar c_{s_3}$ continuam bastante negativos. Escolhemos $s_2$ como variável entrante. O fluxo ao longo do arco correspondente atinge 14 unidades, e a variável $p_6$ sai da base.

---

### Iterações intermediárias

O algoritmo segue iterando, alternando entre adicionar arcos com custo reduzido negativo (entrantes) e remover arcos cujo fluxo cai a zero (saintes). A cada iteração:

1. Recalcula-se o vetor dual $\mathbf{y}$ a partir da nova árvore.
2. Verifica-se se algum arco fora da árvore tem $\bar c_{ij} < 0$.
3. Em caso afirmativo, escolhe-se o arco entrante, identifica-se o ciclo formado, e ajusta-se o fluxo.

Caso contrário (todos os $\bar c_{ij} \geq 0$), a solução atual é **ótima**.

---

### Iteração 10 (final)

O algoritmo termina após **dez iterações**. As variáveis básicas da solução ótima satisfazem:

| Variável | Valor | Variável | Valor |
|:---:|:---:|:---:|:---:|
| $p_0$ | 85 | $p_1$ | 23 |
| $p_2$ | 14 | $p_4$ | 3 |
| $f_1$ | 19 | $f_2$ | 18 |
| $f_3$ | 18 | $f_4$ | 14 |
| $f_5$ | 14 | $s_3$ | 1 |
| $h_1$ | 4  | $h_4$ | 7 |
| $h_5$ | 11 | $h_6$ | 25 |
| $h_7$ | 40 | | |

**Composição do custo ótimo:**

- Compras de novos: $40 \times \$3{,}00 = \$120{,}00$
- Lavagem rápida: $83 \times \$0{,}75 = \$62{,}50$
- Lavagem lenta: $1 \times \$0{,}50 = \$0{,}50$
- **Custo total ótimo: \$182,75**

**Árvore ótima:**

```mermaid
graph LR
    O0(("O₀"))
    O1(("O₁"))
    O2(("O₂"))
    O3(("O₃"))
    O4(("O₄"))
    O5(("O₅"))
    O6(("O₆"))
    O7(("O₇"))

    D0(("D₀"))
    D1(("D₁"))
    D2(("D₂"))
    D3(("D₃"))
    D4(("D₄"))
    D5(("D₅"))
    D6(("D₆"))
    D7(("D₇"))

    %% Compras (apenas p1, p2, p4 e p0)
    O0 -- "p₁=23" --> D1
    O0 -- "p₂=14" --> D2
    O0 -- "p₄=3"  --> D4
    O0 -- "p₀=85" --> D0

    %% Lavagem rápida
    O1 -- "f₁=19" --> D3
    O2 -- "f₂=18" --> D4
    O3 -- "f₃=18" --> D5
    O4 -- "f₄=14" --> D6
    O5 -- "f₅=14" --> D7

    %% Lavagem lenta
    O3 -- "s₃=1" --> D7

    %% Retenção (apenas h1, h4, h5, h6, h7)
    O1 -- "h₁=4"  --> O2
    O4 -- "h₄=7"  --> O5
    O5 -- "h₅=11" --> O6
    O6 -- "h₆=25" --> O7
    O7 -- "h₇=40" --> D0

    classDef fonte fill:#cde4ff,stroke:#1e40af,stroke-width:2px,color:#000;
    classDef sumidouro fill:#ffe4cd,stroke:#b45309,stroke-width:2px,color:#000;
    class O0,O1,O2,O3,O4,O5,O6,O7 fonte;
    class D0,D1,D2,D3,D4,D5,D6,D7 sumidouro;
```

> Observe como a estrutura da árvore ótima difere radicalmente da inicial: agora a maior parte do fluxo passa pelos arcos de lavagem rápida ($f_1, \dots, f_5$), e quase nada é retido até o fim. As variáveis $p_3, p_5, p_6, p_7$ (compras nos dias 3 a 7) saíram da base, sendo substituídas por guardanapos lavados que chegam da lavanderia.

---

### Tabela 2 — Cronograma ótimo

| Dia | Chegam da lavanderia | Comprados | → Rápida | → Lenta | Carregar adiante |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | — | 23 | 19 | — | 4 |
| 2 | — | 14 | 18 | — | — |
| 3 | 19 | — | 18 | 1 | — |
| 4 | 18 | 3 | 14 | — | 7 |
| 5 | 18 | — | 14 | — | 11 |
| 6 | 14 | — | — | — | 24 |
| 7 | 15 | — | — | — | — |

> O fato de **todos** os guardanapos dos dias 1 e 2 serem comprados é esperado: como o caterer começa sem estoque e o serviço rápido leva já dois dias, não há guardanapos limpos disponíveis nesses dias. Se ele já tivesse guardanapos no início do dia 1 (até 37), esse número seria subtraído das compras dos dias 1 e 2.

---

### Verificação no Notebook

A modelagem implementada com OR-Tools/SCIP usa branch-and-cut sobre o modelo de PLI, não o simplex em rede diretamente. Ainda assim, como o problema é uma rede de transbordo com matriz totalmente unimodular, ambos os métodos chegam à mesma solução ótima inteira.

Convertendo a notação do livro para a do notebook:

| Variável do livro | Variável no notebook | Valor |
|:---:|:---:|:---:|
| $p_1$ | `x_1_n` | 23 |
| $p_2$ | `x_2_n` | 14 |
| $p_4$ | `x_4_n` | 3 |
| $f_1$ | `x_1_f` | 19 |
| $f_2$ | `x_2_f` | 18 |
| $f_3$ | `x_3_f` | 18 |
| $f_4$ | `x_4_f` | 14 |
| $f_5$ | `x_5_f` | 14 |
| $s_3$ | `x_3_s` | 1 |
| $h_1$ | `x_1_r` | 4 |
| $h_4$ | `x_4_r` | 7 |
| $h_5$ | `x_5_r` | 11 |
| $h_6$ | `x_6_r` | 25 |

Todas as demais variáveis do notebook (`x_3_n`, `x_5_n`, `x_6_n`, `x_7_n`, `x_1_s`, `x_2_s`, `x_2_r`, `x_3_r`, `x_7_r`) devem assumir valor **zero** na solução ótima.

As variáveis $p_0 = 85$ e $h_7 = 40$ do modelo do livro **não aparecem explicitamente** no notebook: como já discutido, elas correspondem aos arcos auxiliares de balanço da rede ($O_0 \to D_0$ e $O_7 \to D_0$), que não fazem parte da formulação local adotada na implementação.

Ao executar `solver.Solve()` no notebook, espera-se obter exatamente o mesmo custo total: **US$ 182,75**.

## Análise de Sensibilidade

A análise de sensibilidade investiga **como a solução ótima reage a perturbações nos dados de entrada** — preços, demandas, ou o tamanho do estoque do fornecedor. Esse tipo de análise é fundamental para responder perguntas práticas como: *"e se o preço do guardanapo aumentar 20%?"* ou *"até quanto a demanda pode crescer sem mudar o plano?"*.

Como o modelo do catering possui **apenas restrições de igualdade**, **não faz sentido alterar um único valor do lado direito** isoladamente — isso tornaria o problema inviável. Para mexer na demanda $d_i$, é necessário alterar simultaneamente o valor positivo (oferta de sujos em $O_i$) e o valor negativo (demanda em $D_i$).

---

### Variação do estoque do fornecedor de guardanapos

Atualmente, $O_0$ tem **125** guardanapos disponíveis. Quão pequeno esse estoque pode ser **sem alterar o cronograma ótimo**?

O mínimo é **40 guardanapos**, pelas seguintes razões:

- **Dias 1 e 2:** são comprados $23 + 14 = 37$ guardanapos. Como o caterer começa sem estoque e a lavagem rápida leva 2 dias, não há alternativa.
- **Dia 4:** mesmo se todos os guardanapos usados nos dias 1 e 2 forem enviados à lavagem rápida (retornando nos dias 3 e 4), ainda faltam 3 guardanapos no dia 4 — que precisam ser comprados.

Total mínimo: $23 + 14 + 3 = 40$.

**No outro extremo**, se o caterer optasse por **comprar todos** os guardanapos (sem lavagem), o fornecedor precisaria de:

$$
23 + 14 + 19 + 21 + 18 + 14 + 15 = 124 \text{ guardanapos}
$$

> **Conexão com o notebook:** o estoque de $O_0$ não aparece como variável explícita no modelo do notebook — a soma das compras `x_i_n` é livre. Como o solver minimiza custos, ele naturalmente compra o mínimo necessário (40), e o "excesso" do estoque virtual do fornecedor fica implícito.

---

### Variação da demanda

#### Caso 1 — Demanda do dia 1 ($d_1$)

Uma mudança em $d_1$ afeta diretamente $p_1$ (compras no dia 1), já que não há lavagens retornando nesse dia. A propagação para os dias seguintes pode ser resumida assim:

| Faixa de $d_1$ | $p_1$ | $p_2$ | $p_3$ | $p_4$ |
|:---:|:---:|:---:|:---:|:---:|
| $0 \leq d_1 \leq 19$ | $d_1$ | 14 | $19 - d_1$ | 7 |
| $19 \leq d_1 \leq 26$ | $d_1$ | 14 | 0 | $26 - d_1$ |
| $d_1 \geq 26$ | $d_1$ | 14 | 0 | 0 |

**Interpretação:**

- Quando $d_1$ cresce, mais guardanapos são lavados e retornam no dia 3, reduzindo a necessidade de compras nesse dia.
- A partir de $d_1 = 19$, o dia 3 já é totalmente coberto por lavagens — não é mais necessário comprar nada nesse dia.
- A partir de $d_1 = 26$, até o dia 4 fica coberto.

#### Caso 2 — Demanda do dia 3 ($d_3$)

A função de perturbação para $d_3 = 19 + \varepsilon$ é:

| $d_3$ | Custo total $z^*$ |
|:---:|:---:|
| 0  | \$162,00 |
| 9  | \$171,00 |
| 18 | \$180,00 |
| 19 | \$182,75 (atual) |
| 29 | \$210,25 |

O comportamento é **piecewise linear** (linear por partes) com uma "dobra" próxima a $d_3 = 18$:

- Para $0 \leq d_3 \leq 18$: o número total de compras permanece **constante em 39** guardanapos. O custo cresce lentamente, refletindo apenas a mudança nas lavagens.
- Para $d_3 \geq 18$: o caterer precisa **comprar mais** guardanapos (passa para 46 quando $d_3 = 25$). O custo cresce mais rapidamente.

Esse tipo de função em que a inclinação muda em pontos discretos é típica de problemas lineares: a estrutura da solução ótima (quais variáveis estão na base) muda em cada "kink", e a inclinação corresponde ao custo marginal naquela região.

> **Conexão com o notebook:** o notebook pode ser usado para verificar essa análise empiricamente. Basta modificar o dicionário `DEMANDAS` (por exemplo, `DEMANDAS[3] = 25`) e reexecutar; o custo total impresso deve bater com a tabela acima.

---

### Variação dos preços de compra e lavagem

#### Região de tolerância para os custos de lavagem

Sejam $F$ e $S$ os custos de lavagem rápida e lenta, respectivamente (atualmente $F = 0{,}75$ e $S = 0{,}50$). A questão é: **dentro de que faixas de $F$ e $S$ o cronograma ótimo atual permanece inalterado?**

A resposta é a região definida pelo sistema:

$$
F \leq S \leq \tfrac{3}{2}(F - 1)
$$

Algumas implicações práticas:

- A lavanderia pode aumentar o preço do **serviço rápido até US\$ 1,33** mantendo o cronograma atual.
- O preço do **serviço lento pode ir até US\$ 0,75**.
- Pontos extremos dessa região (com custos ótimos correspondentes):
  - $(F, S) = (0, 0)$: $z^* = \$120{,}00$ (só compras necessárias)
  - $(F, S) = (1, 0{,}5)$: $z^* = \$203{,}00$
  - $(F, S) = (1{,}5, 1{,}5)$: $z^* = \$246{,}00$
  - $(F, S) = (3, 3)$: $z^* = \$372{,}00$ (lavar fica tão caro que vale só comprar)

#### Como deduzir a região algebricamente

Variar $F$ e $S$ afeta apenas o **vetor de custos** $\mathbf{c}$. Pelo teorema de otimalidade do simplex, a solução-árvore atual permanece ótima se e somente se **todos os custos reduzidos das variáveis não-básicas são não-negativos**:

$$
\left(\mathbf{c}_N^\top(F, S) - \mathbf{c}_B^\top(F, S)\, \mathbf{B}^{-1}\mathbf{N}\right)_\alpha \geq 0 \quad \text{para toda } \alpha \in NI
$$

onde $\mathbf{B}$ é a matriz básica da solução ótima (contendo as colunas de $p_0, p_1, p_2, p_4, f_1, \dots, f_5, s_3, h_1, h_2, h_3, h_4, h_7$) e $\mathbf{N}$ contém as colunas das não-básicas. Resolvendo esse sistema de inequações chega-se a $F \leq S \leq \tfrac{3}{2}(F - 1)$.

#### Função de perturbação para o preço de compra

Para o preço do guardanapo $P$ (atualmente \$3,00):

| Preço $P$ | Custo total $z^*$ | Estratégia |
|:---:|:---:|:---|
| 0,50 | 62,00  | Comprar tudo |
| 0,75 | 81,25  | Comprar quase tudo |
| 1,00 | 99,25  | Mix |
| 1,25 | 112,75 | Mix (transição) |
| 2,00 | 142,75 | Lavar mais |
| **3,00** | **182,75** | **Ótimo atual** |
| 4,00 | 222,75 | Lavar quase tudo |

**Observações:**

- O preço atual de **\$3,00 já é "alto"**: o caterer compra o mínimo possível (40 guardanapos). Aumentar mais o preço não muda o cronograma.
- O caterer só começa a **comprar mais** quando o preço **cai abaixo de \$1,25**.
- Para o **fornecedor**, baixar o preço pode ser ruim: a receita atual é $40 \times \$3{,}00 = \$120{,}00$. Se reduzisse para \$1,25, mesmo vendendo mais guardanapos, a receita cairia para cerca de \$67,50.

> **Conexão com o notebook:** essa análise também pode ser verificada empiricamente. Altere a constante `VALOR_COMPRA = 3` para outros valores (por exemplo, 1.25) e observe como o cronograma e o custo total mudam. Da mesma forma, `VALOR_LAVAGEM_RAPIDA` e `VALOR_LAVAGEM_LENTA` podem ser ajustados para explorar a região de tolerância.

---

### Resumo prático

A análise de sensibilidade do problema do catering revela uma estrutura típica de problemas lineares:

1. **Existe uma "zona de estabilidade"** ao redor da solução ótima, na qual perturbações pequenas nos parâmetros não alteram o cronograma — apenas o custo total muda linearmente.
2. **Fora dessa zona**, a estrutura da solução muda: variáveis entram ou saem da base, e o custo total passa a crescer ou decrescer em outra taxa.
3. **Pontos de quebra (kinks)** das funções de perturbação correspondem a transições entre diferentes soluções-árvore ótimas — exatamente o que o algoritmo simplex em rede percorreria se o parâmetro fosse alterado gradualmente.

Para o problema do catering com os valores atuais:

| Parâmetro | Faixa de estabilidade |
|---|---|
| Estoque do fornecedor | $\geq 40$ (mínimo) ou $\geq 124$ (máximo necessário) |
| Custo da lavagem rápida $F$ | até \$1,33 |
| Custo da lavagem lenta $S$ | até \$0,75 |
| Preço do guardanapo $P$ | $\geq \$1{,}25$ (acima disso, cronograma não muda) |